# Unified 32K Training + Evaluation Notebook (MongoDB)

This notebook is fully self-contained.
It trains (or loads) a DistilRoBERTa model from MongoDB data, evaluates it,
generates plots, and writes metrics/report artifacts.

No external script files are required.


In [ ]:
# ==========================
# Configuration
# ==========================
THEME = "twitter"
# Allowed: twitter, amazon_reviews, social_mixed, steam_ubisoft, all_sources

MONGO_URI = "mongodb://localhost:27017"
MONGO_DB = "bd_team_normalized"
MONGO_COLLECTION = "normalized_reviews"

TRAIN_SIZE = 32000
EVAL_SIZE = 6400
RANDOM_SEED = 42

BASE_MODEL_NAME = "distilroberta-base"
USE_SPARK = True
FORCE_RETRAIN = False
USE_PRETRAINED_IF_EXISTS = True

# analysis/report knobs
QUALITY_SAMPLE_SIZE = 100000
WORDCLOUD_SAMPLE_SIZE = 20000
HASHTAG_SAMPLE_SIZE = 200000
TOP_HASHTAGS = 20


In [ ]:
import inspect
import json
import random
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from datasets import Dataset
from pymongo import MongoClient
from sklearn.metrics import (
    accuracy_score,
    auc,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_curve,
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)
from wordcloud import WordCloud

print("Imports OK")


In [ ]:
# ==========================
# Paths (repo-local)
# ==========================
cwd = Path.cwd().resolve()
if (cwd / "requirements.txt").exists() and (cwd / "notebooks").exists():
    ROOT_DIR = cwd
elif (cwd.parent / "requirements.txt").exists() and (cwd.parent / "notebooks").exists():
    ROOT_DIR = cwd.parent
else:
    raise FileNotFoundError(
        "Notebook must be executed from Rendu/ or Rendu/notebooks/."
    )

WEIGHTS_DIR = ROOT_DIR / "weights"
OUTPUTS_DIR = ROOT_DIR / "outputs"

ALLOWED = {"twitter", "amazon_reviews", "social_mixed", "steam_ubisoft", "all_sources"}
if THEME not in ALLOWED:
    raise ValueError(f"Invalid THEME={THEME}. Allowed: {sorted(ALLOWED)}")

SOURCE_ARG = "twitter" if THEME == "all_sources" else THEME
IS_ALL_SOURCES = THEME == "all_sources"

MODEL_DIR = WEIGHTS_DIR / f"{THEME}_distilroberta" / "hf_model"
THEME_OUTPUTS = OUTPUTS_DIR / THEME

MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
THEME_OUTPUTS.mkdir(parents=True, exist_ok=True)

print("ROOT_DIR:", ROOT_DIR)
print("THEME:", THEME)
print("SOURCE_ARG:", SOURCE_ARG)
print("MODEL_DIR:", MODEL_DIR)
print("OUTPUTS:", THEME_OUTPUTS)


In [ ]:
# ==========================
# Helper functions
# ==========================
URL_RE = re.compile(r"http\S+|www\S+|https\S+", flags=re.MULTILINE)
MENTION_RE = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#\w+")
HASHTAG_EXTRACT_RE = re.compile(r"#([A-Za-z0-9_]+)")
MULTI_SPACE_RE = re.compile(r"\s+")


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def clean_text_for_model(text: str) -> str:
    text = (text or "").strip()
    text = URL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = MULTI_SPACE_RE.sub(" ", text).strip()
    return text


def clean_text_for_wordcloud(text: str) -> str:
    text = (text or "").lower().strip()
    text = URL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = text.replace("#", " ")
    text = MULTI_SPACE_RE.sub(" ", text).strip()
    return text


def base_match(source: str, all_sources: bool) -> dict[str, Any]:
    m: dict[str, Any] = {
        "label_binary": {"$in": [0, 1]},
        "text": {"$type": "string", "$ne": ""},
    }
    if not all_sources:
        m["source"] = source
    return m


def sample_docs(coll, match_filter: dict[str, Any], size: int) -> list[dict[str, Any]]:
    if size <= 0:
        return []
    pipeline = [
        {"$match": match_filter},
        {"$sample": {"size": size}},
        {"$project": {"_id": 1, "text": 1, "label_binary": 1}},
    ]
    return list(coll.aggregate(pipeline, allowDiskUse=True))


def project_docs(coll, match_filter: dict[str, Any]) -> list[dict[str, Any]]:
    pipeline = [
        {"$match": match_filter},
        {"$project": {"_id": 1, "text": 1, "label_binary": 1}},
    ]
    return list(coll.aggregate(pipeline, allowDiskUse=True))


def fetch_balanced_80_20_docs(
    coll,
    source: str,
    all_sources: bool,
    train_size: int,
    eval_size: int,
    seed: int,
):
    if train_size <= 0 or eval_size <= 0:
        raise RuntimeError("train_size and eval_size must be > 0")
    if train_size % 2 != 0 or eval_size % 2 != 0:
        raise RuntimeError("train_size and eval_size must be even for balanced sampling")

    per_label_train = train_size // 2
    per_label_eval = eval_size // 2
    rng = random.Random(seed)

    match_filter = base_match(source, all_sources=all_sources)
    train_docs, eval_docs = [], []
    class_stats = {}

    for label in (0, 1):
        label_match = dict(match_filter)
        label_match["label_binary"] = label
        available = int(coll.count_documents(label_match))
        if available <= 0:
            raise RuntimeError(f"No documents available for label={label}")

        need_total = per_label_train + per_label_eval
        oversample_used = available < need_total

        if not oversample_used:
            sampled = sample_docs(coll, label_match, need_total)
            rng.shuffle(sampled)
            train_part = sampled[:per_label_train]
            eval_part = sampled[per_label_train:per_label_train + per_label_eval]
        else:
            all_docs = project_docs(coll, label_match)
            train_part = [rng.choice(all_docs) for _ in range(per_label_train)]
            eval_part = [rng.choice(all_docs) for _ in range(per_label_eval)]

        train_docs.extend(train_part)
        eval_docs.extend(eval_part)
        class_stats[str(label)] = {
            "available_docs": available,
            "oversample_used": oversample_used,
            "train_docs_target": per_label_train,
            "eval_docs_target": per_label_eval,
        }

    rng.shuffle(train_docs)
    rng.shuffle(eval_docs)

    counts = {
        "train_docs_fetched": len(train_docs),
        "eval_docs_fetched": len(eval_docs),
        "balanced_sampling": True,
        "balanced_train_per_label": per_label_train,
        "balanced_eval_per_label": per_label_eval,
        "balanced_class_stats": class_stats,
    }
    return train_docs, eval_docs, counts


def ensure_spark_session(use_spark: bool, master="local[*]", app_name="HFPipeline"):
    if not use_spark:
        return None
    try:
        from pyspark.sql import SparkSession
    except Exception as exc:
        raise RuntimeError("PySpark required when USE_SPARK=True") from exc

    return (
        SparkSession.builder.master(master)
        .appName(app_name)
        .config("spark.ui.showConsoleProgress", "false")
        .getOrCreate()
    )


def docs_to_hf_dataset(docs: list[dict[str, Any]], spark=None) -> Dataset:
    rows = []
    for d in docs:
        t = d.get("text", "")
        y = d.get("label_binary")
        if isinstance(t, str) and y in (0, 1):
            rows.append({"text": t, "label": int(y)})

    if not rows:
        raise RuntimeError("No valid rows before preprocessing.")

    if spark is None:
        cleaned_rows = []
        for r in rows:
            c = clean_text_for_model(r["text"])
            if c:
                cleaned_rows.append({"text": c, "label": r["label"]})
        if not cleaned_rows:
            raise RuntimeError("No valid rows after preprocessing.")
        return Dataset.from_pandas(pd.DataFrame(cleaned_rows), preserve_index=False)

    from pyspark.sql import functions as F

    sdf = spark.createDataFrame(rows)
    sdf = sdf.withColumn("text", F.regexp_replace(F.col("text"), r"http\\S+|www\\S+|https\\S+", " "))
    sdf = sdf.withColumn("text", F.regexp_replace(F.col("text"), r"@\\w+", " "))
    sdf = sdf.withColumn("text", F.regexp_replace(F.col("text"), r"\\s+", " "))
    sdf = sdf.withColumn("text", F.trim(F.col("text")))
    sdf = sdf.filter((F.length(F.col("text")) > 0) & F.col("label").isin([0, 1]))
    pdf = sdf.select("text", "label").toPandas()
    if pdf.empty:
        raise RuntimeError("No valid rows after Spark preprocessing.")
    pdf["label"] = pdf["label"].astype(int)
    return Dataset.from_pandas(pdf, preserve_index=False)


def build_training_args(output_dir: str, seed: int):
    kwargs = dict(
        output_dir=output_dir,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3.0,
        weight_decay=0.01,
        save_strategy="epoch",
        load_best_model_at_end=True,
        seed=seed,
        report_to=[],
        logging_steps=50,
        save_total_limit=2,
    )
    sig = inspect.signature(TrainingArguments.__init__)
    if "eval_strategy" in sig.parameters:
        kwargs["eval_strategy"] = "epoch"
    else:
        kwargs["evaluation_strategy"] = "epoch"
    return TrainingArguments(**kwargs)


def fetch_eval_docs_sample(coll, source: str, all_sources: bool, eval_size: int) -> list[dict]:
    pipeline = [
        {"$match": base_match(source, all_sources=all_sources)},
        {"$sample": {"size": int(eval_size)}},
        {"$project": {"_id": 0, "text": 1, "label_binary": 1}},
    ]
    return list(coll.aggregate(pipeline, allowDiskUse=True))


def preprocess_eval_docs(docs: list[dict], use_spark: bool):
    rows = []
    for doc in docs:
        text = doc.get("text")
        label = doc.get("label_binary")
        if isinstance(text, str) and label in (0, 1):
            rows.append({"text": text, "label": int(label)})

    if not rows:
        return [], []

    if not use_spark:
        texts = []
        labels = []
        for row in rows:
            cleaned = clean_text_for_model(row["text"])
            if cleaned:
                texts.append(cleaned)
                labels.append(row["label"])
        return texts, labels

    spark = ensure_spark_session(True, app_name="HFEvalPreprocess")
    try:
        from pyspark.sql import functions as F

        sdf = spark.createDataFrame(rows)
        sdf = sdf.withColumn("text", F.regexp_replace(F.col("text"), r"http\\S+|www\\S+|https\\S+", " "))
        sdf = sdf.withColumn("text", F.regexp_replace(F.col("text"), r"@\\w+", " "))
        sdf = sdf.withColumn("text", F.regexp_replace(F.col("text"), r"\\s+", " "))
        sdf = sdf.withColumn("text", F.trim(F.col("text")))
        sdf = sdf.filter((F.length(F.col("text")) > 0) & F.col("label").isin([0, 1]))
        pdf = sdf.select("text", "label").toPandas()
    finally:
        spark.stop()

    if pdf.empty:
        return [], []
    return pdf["text"].tolist(), pdf["label"].astype(int).tolist()


def evaluate_saved_model(coll, source: str, all_sources: bool, model_path: Path, eval_size: int, use_spark: bool):
    tokenizer = AutoTokenizer.from_pretrained(str(model_path))
    model = AutoModelForSequenceClassification.from_pretrained(str(model_path))

    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")

    model.to(device)
    model.eval()

    eval_docs = fetch_eval_docs_sample(coll, source, all_sources, eval_size)
    texts, y_true = preprocess_eval_docs(eval_docs, use_spark=use_spark)
    if not y_true:
        return {"available": False, "reason": "No evaluation documents found."}

    y_pred, y_score = [], []
    batch_size = 128
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        enc = tokenizer(batch_texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            logits = model(**enc).logits
            probs = torch.softmax(logits, dim=-1)[:, 1]
            preds = torch.argmax(logits, dim=-1)
        y_score.extend(probs.detach().cpu().numpy().tolist())
        y_pred.extend(preds.detach().cpu().numpy().tolist())

    fpr, tpr, _ = roc_curve(y_true, y_score)
    precision_curve, recall_curve, _ = precision_recall_curve(y_true, y_score)

    return {
        "available": True,
        "eval_docs": len(y_true),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": float(auc(fpr, tpr)),
        "pr_auc": float(auc(recall_curve, precision_curve)),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
        "fpr": fpr.tolist(),
        "tpr": tpr.tolist(),
        "precision_curve": precision_curve.tolist(),
        "recall_curve": recall_curve.tolist(),
    }


In [ ]:
# ==========================
# MongoDB connection + balanced 32K/6.4K sampling
# ==========================
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=8000)
client.admin.command("ping")
coll = client[MONGO_DB][MONGO_COLLECTION]
print("MongoDB connection OK")

train_docs, eval_docs, sample_info = fetch_balanced_80_20_docs(
    coll=coll,
    source=SOURCE_ARG,
    all_sources=IS_ALL_SOURCES,
    train_size=TRAIN_SIZE,
    eval_size=EVAL_SIZE,
    seed=RANDOM_SEED,
)
print("Train docs fetched:", len(train_docs))
print("Eval docs fetched :", len(eval_docs))
print(json.dumps(sample_info, indent=2, ensure_ascii=False))


In [ ]:
# ==========================
# Train model (or load pretrained)
# ==========================
used_pretrained = False
train_result_metrics = {}

tokenizer = None
model = None

if MODEL_DIR.exists() and USE_PRETRAINED_IF_EXISTS and not FORCE_RETRAIN:
    print("Loading existing pretrained model from:", MODEL_DIR)
    tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
    model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR))
    used_pretrained = True
else:
    spark = ensure_spark_session(USE_SPARK, app_name="HFTrain32K")
    try:
        train_ds = docs_to_hf_dataset(train_docs, spark=spark)
        eval_ds = docs_to_hf_dataset(eval_docs, spark=spark)
    finally:
        if spark is not None:
            spark.stop()

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

    def preprocess_function(examples):
        return tokenizer(
            examples["text"],
            padding="max_length",
            truncation=True,
            max_length=128,
        )

    tokenized_train = train_ds.map(preprocess_function, batched=True)
    tokenized_eval = eval_ds.map(preprocess_function, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL_NAME, num_labels=2)

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {
            "accuracy": accuracy_score(labels, preds),
            "f1": f1_score(labels, preds, average="binary", zero_division=0),
        }

    train_args = build_training_args(
        output_dir=str(MODEL_DIR / "trainer_output"),
        seed=RANDOM_SEED,
    )

    trainer_kwargs = dict(
        model=model,
        args=train_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_eval,
        compute_metrics=compute_metrics,
    )

    trainer_sig = inspect.signature(Trainer.__init__)
    if "processing_class" in trainer_sig.parameters:
        trainer_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in trainer_sig.parameters:
        trainer_kwargs["tokenizer"] = tokenizer

    trainer = Trainer(**trainer_kwargs)
    train_result = trainer.train()
    _eval_metrics = trainer.evaluate()

    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(MODEL_DIR))
    tokenizer.save_pretrained(str(MODEL_DIR))

    train_result_metrics = {
        "train_result": train_result.metrics,
        "trainer_eval": _eval_metrics,
        "train_docs_used": len(train_ds),
        "eval_docs_used": len(eval_ds),
    }
    print("Training finished. Model saved to:", MODEL_DIR)

print("used_pretrained:", used_pretrained)


In [ ]:
# ==========================
# Evaluate model (fresh sampled eval)
# ==========================
model_eval = evaluate_saved_model(
    coll=coll,
    source=SOURCE_ARG,
    all_sources=IS_ALL_SOURCES,
    model_path=MODEL_DIR,
    eval_size=EVAL_SIZE,
    use_spark=USE_SPARK,
)

print(json.dumps(model_eval, indent=2, ensure_ascii=False)[:3000])


In [ ]:
# ==========================
# Analytics data extraction
# ==========================
def fetch_weekly_sentiment(coll, source: str, all_sources: bool) -> pd.DataFrame:
    pipeline = [
        {"$match": base_match(source, all_sources=all_sources) | {"created_at": {"$ne": None}}},
        {
            "$project": {
                "week_year": {"$isoWeekYear": "$created_at"},
                "week": {"$isoWeek": "$created_at"},
                "label_binary": 1,
            }
        },
        {
            "$group": {
                "_id": {"week_year": "$week_year", "week": "$week"},
                "positive_rate": {"$avg": "$label_binary"},
                "volume": {"$sum": 1},
            }
        },
        {"$sort": {"_id.week_year": 1, "_id.week": 1}},
    ]
    rows = list(coll.aggregate(pipeline, allowDiskUse=True))
    if not rows:
        return pd.DataFrame(columns=["week_year", "week", "week_label", "week_start", "positive_rate", "volume"])

    out = []
    for r in rows:
        wy = int(r["_id"]["week_year"])
        w = int(r["_id"]["week"])
        try:
            ws = datetime.strptime(f"{wy}-W{w:02d}-1", "%G-W%V-%u")
        except ValueError:
            ws = pd.NaT
        out.append(
            {
                "week_year": wy,
                "week": w,
                "week_label": f"{wy}-W{w:02d}",
                "week_start": ws,
                "positive_rate": r["positive_rate"],
                "volume": r["volume"],
            }
        )
    return pd.DataFrame(out)


def fetch_daily_sentiment(coll, source: str, all_sources: bool) -> pd.DataFrame:
    pipeline = [
        {"$match": base_match(source, all_sources=all_sources) | {"created_at": {"$ne": None}}},
        {
            "$project": {
                "day": {"$dateToString": {"format": "%Y-%m-%d", "date": "$created_at"}},
                "label_binary": 1,
            }
        },
        {
            "$group": {
                "_id": "$day",
                "positive_rate": {"$avg": "$label_binary"},
                "volume": {"$sum": 1},
            }
        },
        {"$sort": {"_id": 1}},
    ]
    rows = list(coll.aggregate(pipeline, allowDiskUse=True))
    if not rows:
        return pd.DataFrame(columns=["day", "positive_rate", "volume"])
    df = pd.DataFrame([{"day": r["_id"], "positive_rate": r["positive_rate"], "volume": r["volume"]} for r in rows])
    df["day"] = pd.to_datetime(df["day"], errors="coerce")
    return df.sort_values("day").reset_index(drop=True)


def fetch_hourly_sentiment(coll, source: str, all_sources: bool) -> pd.DataFrame:
    pipeline = [
        {"$match": base_match(source, all_sources=all_sources) | {"created_at": {"$ne": None}}},
        {
            "$project": {
                "dow": {"$isoDayOfWeek": "$created_at"},
                "hour": {"$hour": "$created_at"},
                "label_binary": 1,
            }
        },
        {
            "$group": {
                "_id": {"dow": "$dow", "hour": "$hour"},
                "positive_rate": {"$avg": "$label_binary"},
                "volume": {"$sum": 1},
            }
        },
        {"$sort": {"_id.dow": 1, "_id.hour": 1}},
    ]
    rows = list(coll.aggregate(pipeline, allowDiskUse=True))
    if not rows:
        return pd.DataFrame(columns=["dow", "dow_label", "hour", "positive_rate", "volume"])
    dow_map = {1: "Mon", 2: "Tue", 3: "Wed", 4: "Thu", 5: "Fri", 6: "Sat", 7: "Sun"}
    return pd.DataFrame(
        [
            {
                "dow": r["_id"]["dow"],
                "dow_label": dow_map.get(r["_id"]["dow"], str(r["_id"]["dow"])),
                "hour": r["_id"]["hour"],
                "positive_rate": r["positive_rate"],
                "volume": r["volume"],
            }
            for r in rows
        ]
    )


def fetch_quality_sample(coll, source: str, all_sources: bool, sample_size: int) -> pd.DataFrame:
    pipeline = [
        {"$match": base_match(source, all_sources=all_sources)},
        {"$sample": {"size": int(sample_size)}},
        {"$project": {"_id": 0, "text": 1, "label_binary": 1}},
    ]
    docs = list(coll.aggregate(pipeline, allowDiskUse=True))
    if not docs:
        return pd.DataFrame(columns=["text", "label_binary", "cleaned", "char_len", "word_len", "has_url", "has_mention", "has_hashtag"])

    df = pd.DataFrame(docs)
    df = df[df["label_binary"].isin([0, 1])].copy()
    df["text"] = df["text"].astype(str)
    df["cleaned"] = df["text"].str.lower().str.strip()
    df["char_len"] = df["text"].str.len()
    df["word_len"] = df["text"].str.split().str.len().fillna(0).astype(int)
    df["has_url"] = df["text"].str.contains(URL_RE, regex=True)
    df["has_mention"] = df["text"].str.contains(MENTION_RE, regex=True)
    df["has_hashtag"] = df["text"].str.contains(HASHTAG_RE, regex=True)
    return df


def fetch_top_hashtags(coll, source: str, all_sources: bool, sample_size: int, top_n: int):
    pipeline = [
        {"$match": base_match(source, all_sources=all_sources)},
        {"$sample": {"size": int(sample_size)}},
        {"$project": {"_id": 0, "text": 1, "label_binary": 1}},
    ]
    docs = list(coll.aggregate(pipeline, allowDiskUse=True))
    if not docs:
        empty = pd.DataFrame(columns=["hashtag", "count"])
        return empty, empty

    rows = []
    for d in docs:
        y = d.get("label_binary")
        t = d.get("text")
        if y not in (0, 1) or not isinstance(t, str):
            continue
        tags = [x.lower() for x in HASHTAG_EXTRACT_RE.findall(t)]
        for tag in tags:
            if len(tag) >= 2:
                rows.append({"label_binary": int(y), "hashtag": tag})

    if not rows:
        empty = pd.DataFrame(columns=["hashtag", "count"])
        return empty, empty

    hdf = pd.DataFrame(rows)
    pos = hdf[hdf["label_binary"] == 1]["hashtag"].value_counts().head(int(top_n)).rename_axis("hashtag").reset_index(name="count")
    neg = hdf[hdf["label_binary"] == 0]["hashtag"].value_counts().head(int(top_n)).rename_axis("hashtag").reset_index(name="count")
    return pos, neg


weekly_df = fetch_weekly_sentiment(coll, SOURCE_ARG, IS_ALL_SOURCES)
daily_df = fetch_daily_sentiment(coll, SOURCE_ARG, IS_ALL_SOURCES)
hourly_df = fetch_hourly_sentiment(coll, SOURCE_ARG, IS_ALL_SOURCES)
quality_df = fetch_quality_sample(coll, SOURCE_ARG, IS_ALL_SOURCES, QUALITY_SAMPLE_SIZE)
top_pos_df, top_neg_df = fetch_top_hashtags(coll, SOURCE_ARG, IS_ALL_SOURCES, HASHTAG_SAMPLE_SIZE, TOP_HASHTAGS)

print("weekly rows:", len(weekly_df), "daily rows:", len(daily_df), "hourly rows:", len(hourly_df), "quality rows:", len(quality_df))


In [ ]:
# ==========================
# Generate figures
# ==========================
sns.set_theme(style="whitegrid")
fig_paths = {}

fig1 = plt.figure(figsize=(12, 5))
if not weekly_df.empty:
    w = weekly_df.sort_values(["week_year", "week"]).copy()
    x = w["week_start"] if w["week_start"].notna().any() else w["week_label"]
    plt.plot(x, w["positive_rate"], linewidth=1.8, color="#1f77b4")
    if w["week_start"].notna().any():
        ax = plt.gca()
        ax.xaxis.set_major_locator(mdates.AutoDateLocator(maxticks=12))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
        fig1.autofmt_xdate()
    else:
        plt.xticks(rotation=60)
else:
    plt.text(0.5, 0.5, "No weekly data available", ha="center", va="center")
    plt.axis("off")
plt.ylim(0, 1)
plt.title("Figure 1 - Weekly sentiment trend")
plt.xlabel("Week")
plt.ylabel("Positive rate")
plt.tight_layout()
p1 = THEME_OUTPUTS / "figure1_weekly_sentiment_trend.png"
fig1.savefig(p1, dpi=160)
plt.close(fig1)
fig_paths["figure1"] = p1.name

fig2 = plt.figure(figsize=(12, 5))
if not daily_df.empty:
    d = daily_df.dropna(subset=["day"]).sort_values("day").copy()
    if not d.empty:
        plt.plot(d["day"], d["positive_rate"], linewidth=1.0, alpha=0.45, color="#6baed6", label="daily")
        if len(d) >= 7:
            d["rolling_7d"] = d["positive_rate"].rolling(window=7, min_periods=1).mean()
            plt.plot(d["day"], d["rolling_7d"], linewidth=2.0, color="#1f77b4", label="7-day mean")
            plt.legend()
        ax = plt.gca()
        ax.xaxis.set_major_locator(mdates.AutoDateLocator(maxticks=12))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
        fig2.autofmt_xdate()
else:
    plt.text(0.5, 0.5, "No daily data available", ha="center", va="center")
    plt.axis("off")
plt.ylim(0, 1)
plt.title("Figure 2 - Daily sentiment trend")
plt.xlabel("Day")
plt.ylabel("Positive rate")
plt.tight_layout()
p2 = THEME_OUTPUTS / "figure2_daily_sentiment_trend.png"
fig2.savefig(p2, dpi=160)
plt.close(fig2)
fig_paths["figure2"] = p2.name

fig3 = plt.figure(figsize=(9, 6))
if not quality_df.empty:
    q = quality_df.copy()
    q["sentiment"] = q["label_binary"].map({0: "negative", 1: "positive"})
    ax = sns.boxplot(
        data=q,
        x="sentiment",
        y="char_len",
        hue="sentiment",
        dodge=False,
        showfliers=False,
        palette=["#d62728", "#2ca02c"],
    )
    if ax.legend_ is not None:
        ax.legend_.remove()
    p95 = float(q["char_len"].quantile(0.95))
    plt.ylim(0, max(10, p95))
plt.title("Figure 3 - Text length distribution by sentiment")
plt.xlabel("Sentiment")
plt.ylabel("Characters")
plt.tight_layout()
p3 = THEME_OUTPUTS / "figure3_length_distribution_by_sentiment.png"
fig3.savefig(p3, dpi=160)
plt.close(fig3)
fig_paths["figure3"] = p3.name

fig4 = plt.figure(figsize=(9, 6))
if not top_pos_df.empty:
    d = top_pos_df.head(15).copy()
    d["hashtag"] = "#" + d["hashtag"].astype(str)
    d = d.sort_values("count", ascending=True)
    sns.barplot(data=d, x="count", y="hashtag", color="#2ca02c")
else:
    plt.text(0.5, 0.5, "No positive hashtags available", ha="center", va="center")
    plt.axis("off")
plt.title("Figure 4 - Top positive hashtags")
plt.xlabel("Count")
plt.ylabel("Hashtag")
plt.tight_layout()
p4 = THEME_OUTPUTS / "figure4_top_positive_hashtags.png"
fig4.savefig(p4, dpi=160)
plt.close(fig4)
fig_paths["figure4"] = p4.name

fig5 = plt.figure(figsize=(9, 6))
if not top_neg_df.empty:
    d = top_neg_df.head(15).copy()
    d["hashtag"] = "#" + d["hashtag"].astype(str)
    d = d.sort_values("count", ascending=True)
    sns.barplot(data=d, x="count", y="hashtag", color="#d62728")
else:
    plt.text(0.5, 0.5, "No negative hashtags available", ha="center", va="center")
    plt.axis("off")
plt.title("Figure 5 - Top negative hashtags")
plt.xlabel("Count")
plt.ylabel("Hashtag")
plt.tight_layout()
p5 = THEME_OUTPUTS / "figure5_top_negative_hashtags.png"
fig5.savefig(p5, dpi=160)
plt.close(fig5)
fig_paths["figure5"] = p5.name

fig6 = plt.figure(figsize=(12, 5))
if not hourly_df.empty:
    day_order = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    pivot = hourly_df.pivot(index="dow_label", columns="hour", values="positive_rate").reindex(day_order).sort_index(axis=1)
    sns.heatmap(pivot, cmap="RdYlGn", vmin=0, vmax=1, cbar_kws={"label": "Positive rate"})
else:
    plt.text(0.5, 0.5, "No temporal data available", ha="center", va="center")
    plt.axis("off")
plt.title("Figure 6 - Sentiment heatmap by weekday and hour")
plt.xlabel("Hour")
plt.ylabel("Weekday")
plt.tight_layout()
p6 = THEME_OUTPUTS / "figure6_weekday_hour_heatmap.png"
fig6.savefig(p6, dpi=160)
plt.close(fig6)
fig_paths["figure6"] = p6.name


def sample_texts_for_wordcloud(label: int, sample_size: int) -> str:
    match = {"label_binary": label, "text": {"$type": "string"}}
    if not IS_ALL_SOURCES:
        match["source"] = SOURCE_ARG
    docs = list(coll.aggregate([
        {"$match": match},
        {"$sample": {"size": int(sample_size)}},
        {"$project": {"_id": 0, "text": 1}},
    ], allowDiskUse=True))
    return " ".join(clean_text_for_wordcloud(d.get("text", "")) for d in docs if d.get("text"))


pos_text = sample_texts_for_wordcloud(1, WORDCLOUD_SAMPLE_SIZE)
neg_text = sample_texts_for_wordcloud(0, WORDCLOUD_SAMPLE_SIZE)

wc_pos = WordCloud(width=1600, height=900, background_color="white", max_words=250, collocations=False).generate(pos_text or "positive sentiment")
wc_neg = WordCloud(width=1600, height=900, background_color="white", max_words=250, collocations=False, colormap="magma").generate(neg_text or "negative sentiment")
p7 = THEME_OUTPUTS / "figure7_wordcloud_positive.png"
p8 = THEME_OUTPUTS / "figure8_wordcloud_negative.png"
wc_pos.to_file(str(p7))
wc_neg.to_file(str(p8))
fig_paths["figure7"] = p7.name
fig_paths["figure8"] = p8.name

if model_eval.get("available"):
    cm = np.array(model_eval["confusion_matrix"])
    fig_cm = plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
    plt.title("Figure 9 - Confusion matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    p9 = THEME_OUTPUTS / "figure9_confusion_matrix.png"
    fig_cm.savefig(p9, dpi=170)
    plt.close(fig_cm)
    fig_paths["figure9"] = p9.name

    fig_roc = plt.figure(figsize=(6, 5))
    plt.plot(model_eval["fpr"], model_eval["tpr"], label=f"AUC={model_eval['roc_auc']:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
    plt.title("Figure 10 - ROC curve")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend(loc="lower right")
    plt.tight_layout()
    p10 = THEME_OUTPUTS / "figure10_roc_curve.png"
    fig_roc.savefig(p10, dpi=170)
    plt.close(fig_roc)
    fig_paths["figure10"] = p10.name

    fig_pr = plt.figure(figsize=(6, 5))
    plt.plot(model_eval["recall_curve"], model_eval["precision_curve"], color="#ff7f0e")
    plt.title(f"Figure 11 - Precision-Recall curve (AUC={model_eval['pr_auc']:.4f})")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.tight_layout()
    p11 = THEME_OUTPUTS / "figure11_pr_curve.png"
    fig_pr.savefig(p11, dpi=170)
    plt.close(fig_pr)
    fig_paths["figure11"] = p11.name

print("Generated figures:")
for k, v in fig_paths.items():
    print("-", k, v)


In [ ]:
# ==========================
# Save JSON outputs
# ==========================
scope_match = {} if IS_ALL_SOURCES else {"source": SOURCE_ARG}

summary = {
    "generated_at_utc": utc_now_iso(),
    "theme": THEME,
    "mongo": {
        "uri": MONGO_URI,
        "db": MONGO_DB,
        "collection": MONGO_COLLECTION,
        "source_scope": "all_sources" if IS_ALL_SOURCES else SOURCE_ARG,
    },
    "run_config": {
        "train_size": TRAIN_SIZE,
        "eval_size": EVAL_SIZE,
        "random_seed": RANDOM_SEED,
        "base_model": BASE_MODEL_NAME,
        "use_spark": USE_SPARK,
        "force_retrain": FORCE_RETRAIN,
        "used_pretrained": used_pretrained,
    },
    "sampling_info": sample_info,
    "dataset_overview": {
        "total_docs_scope": int(coll.count_documents(scope_match)),
        "positive_docs_scope": int(coll.count_documents(scope_match | {"label_binary": 1})),
        "negative_docs_scope": int(coll.count_documents(scope_match | {"label_binary": 0})),
    },
    "train_info": train_result_metrics,
    "model_eval": model_eval,
    "figures": fig_paths,
    "paths": {
        "model_dir": str(MODEL_DIR),
        "outputs_dir": str(THEME_OUTPUTS),
    },
}

(THEME_OUTPUTS / "metrics_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
(THEME_OUTPUTS / "training_metrics.json").write_text(json.dumps({
    "generated_at_utc": utc_now_iso(),
    "theme": THEME,
    "train_info": train_result_metrics,
    "sampling_info": sample_info,
    "model_dir": str(MODEL_DIR),
}, ensure_ascii=False, indent=2), encoding="utf-8")

print("Saved:")
print("-", THEME_OUTPUTS / "metrics_summary.json")
print("-", THEME_OUTPUTS / "training_metrics.json")
